# Structured Output and Format Control

Most LLM applications do not stop at human-readable text — they feed model output into **code**: databases, UIs, workflow engines, and other APIs. That requires **predictable shape**: JSON objects with known keys, tables with fixed columns, or bullet lists that parse reliably.

**Format control** spans a spectrum: natural-language instructions ("reply in JSON"), delimiter conventions, provider **JSON mode**, and **schema-constrained** generation. Stronger constraints improve parse success but reduce creative freedom and sometimes add latency or cost.

This notebook explains why structure matters, common output formats, provider features (OpenAI JSON mode and JSON Schema), validation with Python, and retry strategies when parsing fails.

Prerequisites: **01–05** in this section (especially anatomy and role/system prompts). Optional: **Pydantic** basics in `02. Python Libraries/12. Pydentic/`.

---

## 1. Why Structured Output Matters

### 1.1 From Text to Programs

Free-form prose is fine for chat. Production pipelines need **machine-readable** data:

| Downstream step | Needs |
|-----------------|-------|
| Route support ticket | `category` enum |
| Store extraction | JSON matching DB columns |
| Render UI card | Title, summary, severity fields |
| Call another API | Typed arguments (city, date) |
| Evaluate quality | Scores in a fixed rubric |

If the model wraps JSON in markdown fences, adds commentary, or uses wrong key names, your parser breaks and the pipeline fails.

### 1.2 The Reliability Ladder

From weakest to strongest (generally):

1. **Hope** — no format instruction
2. **Prompt-only** — "Return JSON with keys ..."
3. **Delimiters / templates** — `FINAL_ANSWER:`, XML tags
4. **JSON mode** — provider enforces valid JSON syntax
5. **JSON Schema / strict mode** — provider constrains keys, types, enums
6. **Tool / function calling** — model returns arguments object for a declared function

Choose the **lowest** rung that meets reliability needs. Schema mode costs complexity; prompt-only is enough for prototypes.

### 1.3 Trade-offs

| Stronger structure | Benefit | Cost |
|--------------------|---------|------|
| JSON Schema | Fewer parse errors, typed fields | Schema design, provider support |
| JSON mode | Valid JSON, any keys | Keys may still drift |
| Prompt-only | Flexible, portable | Fragile under edge cases |

---

## 2. Output Formats

### 2.1 JSON (Most Common)

Best for APIs, configs, and nested data. Example target:

```json
{
  "sentiment": "NEGATIVE",
  "confidence": 0.91,
  "topics": ["shipping", "packaging"]
}
```

Prompt tip: show a **one-line example** and say "no markdown, no prose outside JSON."

### 2.2 Markdown Tables and Lists

Useful for human-facing reports and simple CSV-like extraction:

| Product | Score | Note |
|---------|-------|------|
| Widget A | 8 | Fast shipping |

Parsing: split lines, detect header row, split on `|`. Fragile if cells contain `|` characters — prefer JSON for automation.

### 2.3 XML and Tagged Blocks

Some pipelines use explicit tags for extraction:

```xml
<summary>One sentence.</summary>
<category>BILLING</category>
```

Regex or `xml.etree` can parse these. Tags reduce ambiguity vs bare labels.

### 2.4 Fixed-Line Patterns

From notebook 04 — lightweight and easy to regex:

```
CATEGORY: REPLACEMENT
REASON: Item arrived damaged.
```

Good for 2–5 flat fields; does not scale to nested data.

---

## 3. Prompt-Only Format Control

Before enabling provider JSON features, master **clear format instructions** in the user or system message:

```
Extract fields from the review.
Return a single JSON object with exactly these keys:
- sentiment (one of: POSITIVE, NEGATIVE, NEUTRAL)
- summary (string, max 20 words)
Do not include markdown fences or any text outside the JSON object.

Review: ...
```

**Common failures:**

- Preamble: "Here is the JSON:"
- Markdown code fence: ` ```json ... ``` `
- Trailing commentary after the closing `}`
- Wrong key casing (`Sentiment` vs `sentiment`)

Mitigations: low `temperature`, few-shot example of **only** JSON, post-process strip fences (demo below).

---

## 4. Provider Features

### 4.1 OpenAI — JSON Mode

Set `response_format={"type": "json_object"}`. The model must output **parseable JSON**. You still define keys in the prompt; the API does not enforce a schema.

**Requirement:** Include the word **JSON** in messages (system or user) — OpenAI rejects JSON mode otherwise.

### 4.2 OpenAI — Structured Outputs (JSON Schema)

Set `response_format` with `type: json_schema` and a JSON Schema definition. With `strict: true`, the model must match the schema (supported models include `gpt-4o-mini`).

Best when you need **fixed keys, types, and enums** without post-validation guesswork.

### 4.3 Anthropic Claude

Claude does not use OpenAI's `response_format`. Common patterns:

- Strong format instructions in system prompt
- **Tool use** — define a tool with an `input_schema`; model returns structured `tool_use` blocks
- Post-parse with Pydantic

### 4.4 Google Gemini

Gemini supports `response_mime_type: "application/json"` and optional `response_schema` in generation config — analogous to JSON mode + schema. See Gemini docs when wiring multi-provider apps.

### 4.5 Comparison

| Provider | Syntax guarantee | Schema guarantee |
|----------|------------------|------------------|
| OpenAI JSON mode | Valid JSON | No |
| OpenAI JSON Schema | Valid JSON | Yes (strict) |
| Claude (prompt/tools) | No | Via tool schema |
| Gemini JSON config | Valid JSON | Optional schema |

---

## 5. Demonstration Setup

Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import re

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

REVIEW = (
    "Package arrived two days late and the box was crushed, but the product "
    "itself works perfectly and customer service refunded the shipping fee."
)


def strip_json_fences(text: str) -> str:
    """Remove optional ```json ... ``` wrappers from model output."""
    text = text.strip()
    match = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)
    return text


print("Setup complete.")

---

## 6. Demonstration: Prompt-Only JSON (Fragile)

No `response_format` — only instructions. Often works; sometimes adds fences or extra text.

In [ ]:
prompt_only = f"""Analyze the review. Return JSON with keys:
- sentiment: POSITIVE, NEGATIVE, or NEUTRAL
- summary: string under 15 words

Review: {REVIEW}"""

raw = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_only}],
    temperature=0.0,
    max_tokens=150,
).choices[0].message.content

print("Raw output:")
print(raw)
print()

try:
    data = json.loads(strip_json_fences(raw or ""))
    print("Parsed:", data)
except json.JSONDecodeError as e:
    print("Parse failed:", e)

If parse fails, note whether fences or prose caused it — that motivates JSON mode.

---

## 7. Demonstration: OpenAI JSON Mode

Valid JSON guaranteed; keys still come from your prompt.

In [ ]:
json_mode_prompt = f"""Return a JSON object analyzing this review.
Keys: sentiment (POSITIVE|NEGATIVE|NEUTRAL), summary (string), topics (array of strings).

Review: {REVIEW}"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": json_mode_prompt}],
    temperature=0.0,
    max_tokens=200,
    response_format={"type": "json_object"},
)

content = response.choices[0].message.content or "{}"
data = json.loads(content)
print(json.dumps(data, indent=2))
print("\nType of topics:", type(data.get("topics")))

JSON mode removes syntax errors. The model might still invent extra keys or wrong enum values — validate in application code.

---

## 8. Demonstration: Structured Outputs (JSON Schema)

Strict schema: fixed keys, types, and enum for `sentiment`.

In [ ]:
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["POSITIVE", "NEGATIVE", "NEUTRAL"],
        },
        "confidence": {
            "type": "number",
            "description": "Model confidence from 0.0 to 1.0",
        },
        "topics": {
            "type": "array",
            "items": {"type": "string"},
        },
        "summary": {"type": "string"},
    },
    "required": ["sentiment", "confidence", "topics", "summary"],
    "additionalProperties": False,
}

schema_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You extract structured review analysis as JSON.",
        },
        {"role": "user", "content": f"Analyze:\n{REVIEW}"},
    ],
    temperature=0.0,
    max_tokens=200,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "review_analysis",
            "strict": True,
            "schema": REVIEW_SCHEMA,
        },
    },
)

structured = json.loads(schema_response.choices[0].message.content or "{}")
print(json.dumps(structured, indent=2))

Expected: mixed review → often **NEUTRAL** or **NEGATIVE** with topics like shipping, packaging, support. All four required keys present; no extra keys allowed under `strict`.

---

## 9. Validation with Pydantic

Schema mode reduces errors; **application-level validation** still catches business rules and powers clear error messages. Pydantic models mirror your JSON shape:

In [ ]:
try:
    from pydantic import BaseModel, Field
    from typing import Literal

    class ReviewAnalysis(BaseModel):
        sentiment: Literal["POSITIVE", "NEGATIVE", "NEUTRAL"]
        confidence: float = Field(ge=0.0, le=1.0)
        topics: list[str]
        summary: str

    validated = ReviewAnalysis.model_validate(structured)
    print("Validated model:", validated)
    print("Route to queue:", "escalation" if validated.sentiment == "NEGATIVE" else "standard")
except ImportError:
    print("Install pydantic: pip install pydantic")
    print("Raw dict still usable:", structured)

---

## 10. Demonstration: Markdown Table Format

Not every output must be JSON. For reports, specify columns explicitly:

In [ ]:
PRODUCTS = """- Alpha earbuds: $79, 4.2 stars, 120 reviews
- Beta earbuds: $99, 4.6 stars, 890 reviews
- Gamma earbuds: $59, 3.8 stars, 45 reviews"""

table_prompt = f"""Convert to a markdown table with columns: Product | Price | Rating | Reviews.
Sort by rating descending. No extra commentary.

{PRODUCTS}"""

table_out = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": table_prompt}],
    temperature=0.0,
    max_tokens=250,
).choices[0].message.content

print(table_out)

---

## 11. Validation Failures and Retries

### 11.1 What Can Still Go Wrong

| Layer | Failure |
|-------|---------|
| Prompt-only | Invalid JSON, fences |
| JSON mode | Wrong types inside JSON (string instead of number) |
| Schema mode | Rare violations on unsupported models; request errors on bad schema |
| App validation | Business rules (confidence < 0.5, empty topics) |

### 11.2 Retry Pattern

1. Call model with structured settings
2. `json.loads` → on failure, retry with stricter reminder
3. Pydantic `model_validate` → on failure, retry with error details in prompt
4. Cap retries (e.g. 2); fall back to human review or safe default

Always log **raw completion** for debugging.

In [ ]:
def extract_review_analysis(review: str, max_attempts: int = 2) -> dict:
    """JSON mode + parse retry with corrective user message."""
    messages = [
        {
            "role": "system",
            "content": "Return JSON only with keys: sentiment, summary, topics.",
        },
        {"role": "user", "content": f"Analyze:\n{review}"},
    ]
    last_error = None
    for attempt in range(max_attempts):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            max_tokens=200,
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content or ""
        try:
            return json.loads(strip_json_fences(raw))
        except json.JSONDecodeError as e:
            last_error = e
            messages.append({"role": "assistant", "content": raw})
            messages.append(
                {
                    "role": "user",
                    "content": f"Invalid JSON ({e}). Reply with valid JSON only, no markdown.",
                }
            )
    raise ValueError(f"Failed after {max_attempts} attempts: {last_error}")


result = extract_review_analysis(REVIEW)
print(json.dumps(result, indent=2))

With JSON mode, the retry loop rarely triggers — it matters more for prompt-only or cross-provider setups.

---

## 12. Designing Schemas for LLMs

| Guideline | Reason |
|-----------|--------|
| Fewer fields | Less confusion, lower token cost |
| Use `enum` for labels | Prevents synonym drift (`pos` vs `POSITIVE`) |
| `additionalProperties: false` | Stops surprise keys (OpenAI strict) |
| Short `description` on fields | Guides model semantics |
| Avoid deep nesting | Flatter JSON parses and validates easier |
| Match names to your code | `snake_case` vs `camelCase` consistency |

For evolving APIs, version schemas (`review_analysis_v2`) and keep parsers backward compatible.

---

## 13. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| JSON mode without "JSON" in prompt | API error | Mention JSON in messages |
| No validation after schema | Bad business data slips through | Pydantic + rules |
| Huge nested schema | Timeouts, confused model | Split into steps (chaining) |
| Parsing prose with regex only | Brittle | JSON mode or schema |
| Ignoring `max_tokens` | Truncated JSON | Increase limit |
| Same schema for all providers | Integration bugs | Normalize per provider |

---

## 14. Summary

Structured output turns LLM text into **pipeline-ready data**. Start with clear prompt instructions; upgrade to **JSON mode** for syntax safety and **JSON Schema (strict)** when keys, types, and enums must be reliable.

Validate with `json.loads` and **Pydantic**; retry with corrective messages on failure; log raw completions. Markdown tables and tagged formats suit human-readable outputs; JSON suits automation.

Demonstrations compared prompt-only JSON, JSON mode, strict schema extraction, markdown tables, and a retry helper.

Next: **07. Prompt Chaining and Task Decomposition** — breaking complex work into multiple LLM steps.